# Sesión 1 · Fundamentos del LLM y ventanas de contexto

**AI Engineer I: Fundamentos de IA Generativa** · Módulo 1 · Edutek

---

## 🎯 Objetivos de la sesión

Al terminar este notebook podrás:

1. **Ubicar a los LLM en la historia de la IA** (de la IA clásica a hoy) y explicar qué hace un LLM: predecir el siguiente token.
2. **Explicar qué es un token y una ventana de contexto**, por qué ambos cuestan dinero y calcular el costo de una llamada.
3. **Explicar por qué un LLM no se prueba como software tradicional**: es probabilístico, y los parámetros de generación controlan cuánto.

## 🗺️ Mapa del notebook

| Parte | Tema | ¿Consume dinero? |
| --- | --- | --- |
| 0 | Preparar el entorno | No |
| Prólogo | Breve historia: de la IA clásica a los LLM | No |
| 1 | Qué es un LLM: predecir el siguiente token | No |
| 2 | Tokens y tokenizadores | No (*) |
| 3 | Del token al costo | No |
| 4 | La ventana de contexto | No |
| 5 | Tu primera llamada a la API | **Sí, centavos** |
| 6 | Parámetros de generación | **Sí, centavos** |
| 7 | Determinista vs probabilístico y cuándo NO usar un LLM | No |
| 8 | Cierre, repaso y retos | No |

(*) Una celda opcional de la Parte 2 usa el servicio de conteo de tokens de tu proveedor (en Anthropic es gratuito; en OpenAI confirma en su página de precios). Requiere tu API key.

---
# Parte 0 · Preparar el entorno

Antes de empezar verificamos tres cosas: la versión de Python, las librerías y la API key de tu proveedor (**Anthropic u OpenAI**: el repositorio funciona con ambos).

**¿Cómo se ejecuta un notebook?** Selecciona una celda y pulsa `Shift + Enter`: ejecuta la celda y pasa a la siguiente. Las celdas de texto (como esta) solo se "ejecutan" para renderizarse.

> Si no has creado el entorno, vuelve al `README.md` de esta carpeta: ahí está el paso a paso con `uv`.

In [ ]:
import sys

print("Python:", sys.version.split()[0])
assert sys.version_info[:2] == (3, 12), (
    "Esta sesión está fijada a Python 3.12. Ejecuta `uv sync` dentro de la carpeta "
    "s01-fundamentos-llm y selecciona el kernel .venv (ver README)."
)

### Cargar las variables de entorno

Las llaves de API **nunca** van escritas en el código ni en el notebook. Viven en un archivo `.env` (que está en el `.gitignore`, así que no se sube a Git) o en una variable de entorno.

`find_dotenv()` busca un archivo `.env` subiendo por las carpetas desde donde estás, así que basta con tener **un solo `.env` en la raíz del repositorio**.

### Elegir el proveedor: Anthropic u OpenAI

El repositorio funciona **con los dos proveedores**, y se cambia con una variable de entorno en tu archivo `.env` (raíz del repositorio):

```
LLM_PROVIDER=anthropic        # por defecto
ANTHROPIC_API_KEY=...

# o bien
LLM_PROVIDER=openai
OPENAI_API_KEY=...
```

Opcionalmente puedes fijar el modelo con `LLM_MODEL` (y `LLM_MODEL_SAMPLING` para los experimentos de la Parte 6). Si no los defines, se usa el modelo económico de cada proveedor (`claude-haiku-5-5` o `gpt-6-luna`).

> **Usas otro servicio compatible con OpenAI** (Groq, OpenRouter…)? Define además `OPENAI_BASE_URL`. Ver `docs/PROVEEDORES.md`.

Después de editar el `.env`, **reinicia el kernel** del notebook: las variables se leen al iniciar.

In [ ]:
import os
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))

# cliente_llm.py es el módulo del curso que habla con los proveedores (se explica a fondo en la Sesión 2)
from cliente_llm import (
    PRECIOS, Gasto, costo_usd, hay_api_key, llamar_llm, modelo_economico, modelo_muestreo,
    proveedor_activo, contar_tokens_exacto, variable_api_key,
)

PROVEEDOR = proveedor_activo()          # LLM_PROVIDER en tu .env: "anthropic" (por defecto) u "openai"
HAY_KEY = hay_api_key()
print(f"Proveedor: {PROVEEDOR} · modelo económico: {modelo_economico()}")
print(f"¿API key ({variable_api_key()}) detectada?:", "sí ✅" if HAY_KEY else "no ❌ (puedes seguir; solo se omitirán las celdas de pago)")

### 🔌 Interruptor general de gasto

💰 **FYI:** las celdas de código que llaman a la API de pago de tu proveedor (Anthropic u OpenAI) llevan este emoji en su primera línea, con una estimación de lo que cuestan. Esta variable las controla todas: con `False` no llaman a la API; con `True` y una API key válida, se ejecutan. Te recomendamos tener un límite de gasto mensual configurado en tu cuenta (ver `docs/COSTOS.md`).

In [ ]:
EJECUTAR_LLAMADAS_DE_PAGO = True   # ← ponla en False para no gastar nada

# Las celdas de pago solo corren si el interruptor está activo Y hay key.
PUEDE_GASTAR = EJECUTAR_LLAMADAS_DE_PAGO and HAY_KEY
print("Celdas de pago activas:", PUEDE_GASTAR)

### El contador de gasto

Pequeña utilidad que usaremos todo el curso (vive en `cliente_llm.py`): guarda el resultado de cada llamada, que incluye los tokens de entrada y salida que informa el proveedor, y calcula cuánto costó con la tabla de precios `PRECIOS` (USD por **millón** de tokens).

**Importante:** la API cobra por tokens, y los precios son **por millón**. Por eso una llamada pequeña cuesta fracciones de centavo.

> Los precios son una referencia a octubre de 2026 y cambian. Si usas un modelo que no está en `PRECIOS`, agrégalo (o su costo se contará como 0).

In [ ]:
gasto = Gasto()   # acumula el costo real de las llamadas de pago de este notebook

print(f"{'modelo':<20}{'USD/M entrada':>16}{'USD/M salida':>16}")
for modelo, (pin, pout) in PRECIOS.items():
    print(f"{modelo:<20}{pin:>16.2f}{pout:>16.2f}")
print("\nContador de gasto listo. Gasto actual: US$", gasto.total)

---
# Prólogo · Breve historia de la IA generativa

> *De la IA clásica a los LLM de hoy.* Sin costo: es lectura y un experimento local.

## ¿Por qué empezar por la historia?

Porque entender **de dónde viene** un LLM te ayuda a entender **qué es** y **qué no es**. Tres ideas para llevarte:

1. **Cada ola resolvió el límite de la anterior.** Nada llegó de la nada.
2. **Las ideas son más viejas que la tecnología.** Predecir la siguiente palabra se hacía hace décadas; lo que cambió es la **escala** (datos, cómputo) y la **arquitectura**.
3. **La IA tuvo ciclos de entusiasmo y de decepción** (los llamados *inviernos de la IA*). Por eso este curso es honesto: se enseña lo que funciona y se dice lo que no.

> ⚠️ Las fechas y los hitos de abajo son una **línea de tiempo simplificada** para dar contexto. Las verificamos antes de clase; si alguna te interesa de verdad, consúltala en fuentes primarias.

## Las cinco eras

| Era | Período | Idea central | Hitos | Su límite |
| --- | --- | --- | --- | --- |
| **1 · IA simbólica** | 1950 – 1980s | Programar el conocimiento como **reglas** escritas a mano | Turing y su "prueba" (1950) · Conferencia de Dartmouth, donde nace el término *inteligencia artificial* (1956) · ELIZA, un chatbot de patrones (1966) · **Sistemas expertos** (años 70-80) | Frágil: nadie puede escribir todas las reglas del mundo. Los *inviernos de la IA* llegan cuando las promesas no se cumplen |
| **2 · Aprendizaje estadístico** | 1990 – 2010 | En vez de escribir reglas, **aprender de datos** | Modelos de lenguaje de **n-gramas** · filtros de spam · traducción estadística · Deep Blue vence a Kasparov (1997; búsqueda y fuerza bruta, no aprendizaje) · LSTM (1997) | Las características las diseñaba una persona; poco contexto |
| **3 · Deep learning** | 2012 – 2016 | Redes neuronales **profundas** que aprenden sus propias características, con GPUs | AlexNet gana ImageNet (2012) · **word2vec**: palabras como vectores (2013) · modelos secuencia-a-secuencia y **atención** (2014) · GANs (2014) · AlphaGo (2016) | Procesan el texto en orden, lento de entrenar y con poca memoria de contexto largo |
| **4 · Transformer y escala** | 2017 – 2021 | **Atención** en paralelo + preentrenar con cantidades enormes de texto | *Attention Is All You Need* presenta el **transformer** (2017) · GPT y BERT (2018) · GPT-2 (2019) · **GPT-3**, 175 mil millones de parámetros, y las *leyes de escala* (2020) | Modelos potentes pero difíciles de usar: hay que "dirigirlos" con trucos |
| **5 · Asistentes y agentes** | 2022 – hoy | Alinear al modelo con instrucciones humanas y **ponerlo a disposición por API** | InstructGPT y **RLHF** (2022) · Stable Diffusion y DALL·E 2 (2022) · **ChatGPT**, noviembre de 2022 · GPT-4, Claude y Llama con pesos abiertos (2023) · ventanas de contexto largas · **modelos de razonamiento** (2024-2025) · uso de herramientas, **MCP** y agentes | Alucinaciones, costo, seguridad, evaluación: **el trabajo del AI Engineer** |

## Qué cambió de verdad

- De modelos que hacían **una sola tarea** (traducir, clasificar) a modelos de **propósito general**: el mismo modelo resume, traduce, programa y clasifica.
- De "entrenar tu propio modelo" a **consumirlo por API** como un servicio. Eso crea un rol nuevo, **el AI Engineer** (Parte 1 de la teoría): construir sistemas **encima** de modelos que otros entrenaron.
- La arquitectura *transformer* hizo posible **escalar**: más datos y más cómputo dieron saltos de capacidad que nadie había anticipado del todo.

## 🧪 Experimento: un modelo de lenguaje de los años 90, en 20 líneas

Un **modelo de n-gramas** es el ancestro directo de lo que hace un LLM: dado lo que viene, **predice la siguiente palabra** contando qué palabra suele seguir. Lo construimos con un mini-corpus de texto de TI. **Sin API y sin costo.**

La diferencia clave con un LLM: este modelo solo "ve" las últimas **1 o 2 palabras** (su *orden*); un LLM moderno ve **miles de tokens** y decide qué partes importan (atención).

In [ ]:
import random
from collections import Counter, defaultdict

# Mini-corpus de texto de TI (inventado para la clase)
CORPUS = """
el servidor no responde porque el certificado expiró . el servidor no responde porque el disco está lleno .
el balanceador rechaza las conexiones porque el certificado expiró . el equipo renovó el certificado y reinició el servidor .
el equipo reinició el balanceador y el servicio volvió a responder . el servicio no responde porque el firewall bloquea el puerto .
los usuarios no pueden emitir facturas porque el servicio no responde . los usuarios reportan que el servidor no responde .
el equipo revisó los registros y encontró que el disco está lleno . el equipo renovó el certificado antes de la ventana de mantenimiento .
""".split()


def entrenar(palabras, orden):
    """Cuenta qué palabra sigue a cada grupo de `orden` palabras."""
    modelo = defaultdict(Counter)
    for i in range(len(palabras) - orden):
        contexto = tuple(palabras[i : i + orden])
        modelo[contexto][palabras[i + orden]] += 1
    return modelo


def generar(modelo, orden, inicio, largo=14, semilla=None):
    """Genera texto eligiendo la siguiente palabra según los conteos (¡igual que un LLM, pero con 1-2 palabras de contexto!)."""
    rng = random.Random(semilla)
    texto = list(inicio)
    for _ in range(largo):
        contexto = tuple(texto[-orden:])
        if contexto not in modelo:           # nunca vio este contexto: no sabe qué sigue
            break
        palabras, pesos = zip(*modelo[contexto].items())
        texto.append(rng.choices(palabras, weights=pesos, k=1)[0])
    return " ".join(texto)


for orden in (1, 2, 3):
    m = entrenar(CORPUS, orden)
    print(f"── Contexto de {orden} palabra(s) ──")
    for semilla in (1, 2, 3):
        print("  ", generar(m, orden, CORPUS[:orden], semilla=semilla))

### 🔎 Observa

- Con **1 palabra de contexto** el texto es local pero **incoherente**: cada frase "olvida" cómo empezó.
- Con **3 palabras** suena mucho mejor... porque casi **copia el corpus** (no tiene nada nuevo que decir) y se **detiene** si encuentra un contexto que nunca vio.
- Un LLM resuelve ambos problemas: **contexto largo** (miles de tokens con atención) y **generalización** (no necesita haber visto la frase exacta, porque representa el significado como vectores, ver Sesión 4).

> **La idea de fondo no cambió en 70 años: predecir lo siguiente.** Cambiaron la escala, la arquitectura y la forma de entrenarlo.

### ✍️ Para pensar

¿Qué crees que pasaría si entrenáramos este modelo con **todo internet** pero manteniendo un contexto de 2 palabras? ¿Qué habría que cambiar para que "entienda" una pregunta larga? (Spoiler: eso es lo que hace el transformer.)

---
# Parte 1 · Qué es un LLM: un predictor del siguiente token

## La idea central

Un **modelo de lenguaje grande** (LLM, *Large Language Model*) hace **una sola cosa**: dado un texto, calcula la probabilidad de cada posible continuación, elige una, la agrega al texto y repite.

```
"El servidor no responde porque el ..."
        ↓  el modelo calcula probabilidades
   certificado 34%  ·  disco 18%  ·  servicio 15%  ·  firewall 12% ...
        ↓  elige una (muestreo)
"El servidor no responde porque el certificado ..."
        ↓  repite con el texto nuevo
```

Todo lo que parece "razonamiento" sale de repetir este paso miles de veces.

## Qué hay por dentro (a muy alto nivel)

1. El texto se parte en **tokens** (Parte 2).
2. Cada token se convierte en un **vector** (una lista de números).
3. Varias capas de **atención** deciden qué partes del texto importan para predecir lo que sigue. Esta arquitectura se llama **transformer**.
4. La salida es una **distribución de probabilidad** sobre todos los tokens posibles.

> **Analogía para quien viene de infraestructura:** un LLM no es una base de datos ni un buscador. Es más parecido a un autocompletado muy sofisticado entrenado con enormes cantidades de texto. No "consulta" hechos: produce el texto más plausible. Por eso puede sonar muy seguro y estar equivocado.

## Un LLM de juguete para verlo con tus propios ojos

Vamos a construir, **sin API y sin costo**, el paso de "elegir el siguiente token". Las probabilidades de abajo son **ilustrativas** (las inventamos para la clase); un modelo real las calcula sobre decenas o cientos de miles de tokens posibles.

In [ ]:
import math
import random
from collections import Counter

# Distribución ilustrativa para: "El servidor no responde porque el ..."
PROBABILIDADES = {
    "certificado": 0.34,
    "disco": 0.18,
    "servicio": 0.15,
    "firewall": 0.12,
    "balanceador": 0.09,
    "DNS": 0.06,
    "kernel": 0.04,
    "gato": 0.02,
}
assert abs(sum(PROBABILIDADES.values()) - 1) < 1e-9

for palabra, p in PROBABILIDADES.items():
    print(f"{palabra:<12} {p:>5.0%}  {'█' * int(p * 60)}")

## Los parámetros de generación, explicados con el juguete

Cuando llamas a un modelo puedes ajustar **cómo elige** el siguiente token. Implementémoslo para entenderlo de verdad:

| Parámetro | Qué hace | Analogía |
| --- | --- | --- |
| **temperature** | Reparte o concentra las probabilidades. Baja → casi siempre la más probable. Alta → más variedad (y más disparates). | La "perilla de creatividad". |
| **top_k** | Solo considera las *k* opciones más probables. | Un filtro de "los k mejores candidatos". |
| **top_p** | Solo considera las opciones más probables hasta acumular una probabilidad *p* (por ejemplo 90%). | Un filtro que se adapta: pocas opciones si el modelo está seguro, más si duda. |
| **max_tokens** | Tope de longitud de la respuesta. | Un `timeout`, pero de longitud. Es la forma más directa de **controlar el costo**. |

La temperatura funciona dividiendo los *logits* (la puntuación cruda antes de volverse probabilidad) entre `T` y aplicando `softmax`. Con `T → 0` gana siempre la más probable; con `T` grande todo se vuelve casi uniforme.

In [ ]:
def muestrear(temperature=1.0, top_k=None, top_p=None, n=1, semilla=None):
    """Elige `n` veces el siguiente token del modelo de juguete."""
    rng = random.Random(semilla)
    palabras = list(PROBABILIDADES)
    logits = [math.log(PROBABILIDADES[w]) for w in palabras]

    if temperature == 0:  # decodificación "codiciosa": siempre la más probable
        return [palabras[logits.index(max(logits))]] * n

    # softmax con temperatura
    escalados = [l / temperature for l in logits]
    m = max(escalados)
    exps = [math.exp(l - m) for l in escalados]
    total = sum(exps)
    probs = sorted(zip(palabras, (e / total for e in exps)), key=lambda x: -x[1])

    if top_k is not None:
        probs = probs[:top_k]
    if top_p is not None:
        acumulado, recortado = 0.0, []
        for w, p in probs:
            recortado.append((w, p))
            acumulado += p
            if acumulado >= top_p:
                break
        probs = recortado

    ws, ps = zip(*probs)
    return rng.choices(ws, weights=ps, k=n)


def histograma(titulo, muestras):
    c = Counter(muestras)
    print(f"\n{titulo}")
    for palabra, veces in c.most_common():
        print(f"  {palabra:<12} {veces / len(muestras):>5.0%}  {'█' * int(veces / len(muestras) * 50)}")


N = 2000
histograma("temperature = 0   (siempre la más probable)", muestrear(temperature=0, n=N))
histograma("temperature = 0.3 (conservador)", muestrear(temperature=0.3, n=N, semilla=1))
histograma("temperature = 1.0 (la distribución original)", muestrear(temperature=1.0, n=N, semilla=1))
histograma("temperature = 2.0 (muy creativo… y errático)", muestrear(temperature=2.0, n=N, semilla=1))

### 🔎 Observa

- Con `temperature = 0` **siempre** sale `certificado`. Es lo más cercano a "determinista".
- Con `temperature = 1.0` ves la distribución original: a veces sale `gato` (2%), un disparate.
- Con `temperature = 2.0` hasta las opciones absurdas ganan terreno.

Ahora probemos los recortes `top_k` y `top_p` a temperatura 1.0:

In [ ]:
histograma("top_k = 3 (solo las 3 más probables)", muestrear(temperature=1.0, top_k=3, n=N, semilla=2))
histograma("top_p = 0.8 (hasta acumular 80%)", muestrear(temperature=1.0, top_p=0.8, n=N, semilla=2))

**Idea clave:** `top_k` y `top_p` **recortan la cola** de opciones improbables (como `gato`), así que evitan los disparates sin quitarle toda la variedad al modelo.

> ⚠️ **Dos avisos para el mundo real**
> 1. `temperature = 0` **reduce** la variación, pero **no garantiza** respuestas idénticas (hay no-determinismo en el hardware y en el servicio). Lo comprobaremos en la Parte 6.
> 2. Algunos modelos, sobre todo los de razonamiento y los más recientes, **fijan o rechazan** estos parámetros. También lo veremos en la Parte 6: no asumas que todos los modelos aceptan `temperature`.

### ✍️ Pregunta para pensar

Si quieres **clasificar tickets** de soporte, ¿qué temperatura usarías? ¿Y para **proponer nombres** creativos para un producto? (Respuesta en la Parte 6.)

---
# Parte 2 · Tokens y tokenizadores

## ¿Qué es un token?

El modelo **no ve letras ni palabras**: ve **tokens**, que son fragmentos de texto. Una palabra común suele ser 1 token; una palabra larga, rara o en otro idioma se parte en varios.

Cada modelo (o familia de modelos) trae **su propio tokenizador**, que es la tabla que dice cómo partir el texto. Por eso **el mismo texto da un número distinto de tokens según el modelo**.

## ¿Por qué importa? Tres razones

1. **Se cobra por token**, tanto de entrada como de salida.
2. **El límite del modelo se mide en tokens** (la ventana de contexto, Parte 4).
3. **El idioma pesa:** el español suele necesitar más tokens que el inglés para decir lo mismo, porque los tokenizadores se entrenaron con mucho más texto en inglés.

## 🔮 Antes de ejecutar: haz tu predicción

Vamos a medir tres textos equivalentes en contenido: un párrafo en **español**, el mismo en **inglés** y un bloque de **código**.

> **¿Cuál crees que gastará más tokens?** Anota tu predicción y compárala al final.

Primero, cargamos los textos de ejemplo (un ticket de soporte ficticio del mundo de TI) desde la carpeta `datos/`.

In [ ]:
from pathlib import Path

DATOS = Path("datos")
TEXTO_ES = (DATOS / "ticket_es.txt").read_text(encoding="utf-8")
TEXTO_EN = (DATOS / "ticket_en.txt").read_text(encoding="utf-8")
CODIGO = (DATOS / "log_codigo.txt").read_text(encoding="utf-8")

TEXTOS = {"español": TEXTO_ES, "inglés": TEXTO_EN, "código": CODIGO}
for nombre, t in TEXTOS.items():
    print(f"{nombre:<8} → {len(t):>4} caracteres, {len(t.split()):>3} palabras")

## Tokenizadores que vamos a comparar

Usaremos tres, todos **gratuitos y locales** (se ejecutan en tu computadora):

| Nombre | Librería | Pertenece a |
| --- | --- | --- |
| `o200k_base` | `tiktoken` | Familia de modelos recientes de OpenAI |
| `cl100k_base` | `tiktoken` | Generación anterior de OpenAI |
| `Qwen/Qwen2.5-0.5B` | `tokenizers` (Hugging Face) | Modelo *open-weight* de Alibaba |

> **¿Y los tokenizadores de Claude y de OpenAI?** OpenAI **sí publica** sus tokenizadores (por eso existe `tiktoken`): para sus modelos es la herramienta oficial, aunque el *encoding* exacto de cada modelo se consulta en su documentación. Anthropic **no publica** su tokenizador como librería local: para contar tokens de Claude se usa un endpoint de la API (`count_tokens`), que es **gratuito**. Lo probamos más abajo con el proveedor que hayas elegido. Para Claude, `tiktoken` sirve solo como **aproximación**.

La primera vez, `tiktoken` y `tokenizers` descargan sus tablas de internet (unos MB) y las guardan en caché.

In [ ]:
import tiktoken
from tokenizers import Tokenizer

ENC_O200K = tiktoken.get_encoding("o200k_base")
ENC_CL100K = tiktoken.get_encoding("cl100k_base")
ENC_QWEN = Tokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")   # el aviso sobre HF_TOKEN es normal e inofensivo


TOKENIZADORES = ["o200k_base", "cl100k_base", "Qwen2.5"]


def tokens_de(nombre: str, texto: str) -> list[str]:
    """Devuelve los fragmentos de texto (tokens) según el tokenizador elegido."""
    if nombre == "Qwen2.5":
        # usamos los offsets para recuperar el texto original de cada token
        return [texto[a:b] for a, b in ENC_QWEN.encode(texto).offsets]
    enc = ENC_O200K if nombre == "o200k_base" else ENC_CL100K
    return [enc.decode_single_token_bytes(t).decode("utf-8", errors="replace") for t in enc.encode(texto)]


print("Ejemplo: 'Hola mundo' con o200k_base →", tokens_de("o200k_base", "Hola mundo"))
print("Ejemplo: 'Hola mundo' con Qwen2.5     →", tokens_de("Qwen2.5", "Hola mundo"))

## Ver los tokens

Esta función pinta cada token con un color distinto para que **veas** dónde corta el tokenizador. Fíjate en tres cosas:

- Los **espacios** suelen pegarse al inicio del token siguiente.
- Las palabras comunes son 1 token; las raras se parten en pedazos.
- Los **números** y los **signos** también se fragmentan.

In [ ]:
import html
from IPython.display import HTML, display

COLORES = ["#ffd6d6", "#d6f0ff", "#e0ffd6", "#fff3c4", "#e8d6ff", "#d6fff4"]


def mostrar_tokens(texto: str, tokenizador: str = "o200k_base"):
    piezas = tokens_de(tokenizador, texto)
    spans = []
    for i, p in enumerate(piezas):
        visible = html.escape(p).replace(" ", "&nbsp;").replace("\n", "↵<br>")
        spans.append(
            f'<span style="background:{COLORES[i % len(COLORES)]};color:#111;'
            f'border-radius:3px;padding:1px 0;margin:0 1px;">{visible}</span>'
        )
    display(HTML(f"<p><b>{tokenizador}</b> → {len(piezas)} tokens</p><div style='line-height:2'>{''.join(spans)}</div>"))


frase = "El servidor no responde porque el certificado expiró."
for tk in TOKENIZADORES:
    mostrar_tokens(frase, tk)

### 🔎 Observa

Compara cómo cada tokenizador parte `expiró` y `certificado`. ¿Coinciden? Casi nunca. Cada tokenizador es una "tabla de abreviaturas" distinta aprendida de sus propios datos.

Ahora algunos casos curiosos. Prueba predecir cuántos tokens tendrá cada uno antes de ejecutar:

In [ ]:
CURIOSOS = [
    "servidor",
    "electroencefalografista",
    "2026-10-13T19:30:00-06:00",
    "192.168.100.254",
    "😀🚀",
    "         ",
    "ERROR_CODE_0xDEADBEEF",
]
print(f"{'texto':<28}" + "".join(f"{t:>14}" for t in TOKENIZADORES))
for s in CURIOSOS:
    fila = "".join(f"{len(tokens_de(t, s)):>14}" for t in TOKENIZADORES)
    print(f"{s!r:<28}{fila}")

## La comparación principal: español vs inglés vs código

Ahora respondemos la predicción. La métrica útil es **caracteres por token**: cuanto **más bajo**, peor "eficiencia" (más tokens para el mismo texto, por lo tanto más costo).

In [ ]:
filas = []
for nombre_texto, texto in TEXTOS.items():
    for tk in TOKENIZADORES:
        n = len(tokens_de(tk, texto))
        filas.append((nombre_texto, tk, n, len(texto) / n))

print(f"{'texto':<10}{'tokenizador':<14}{'tokens':>8}{'chars/token':>14}")
print("-" * 46)
for nombre_texto, tk, n, r in filas:
    print(f"{nombre_texto:<10}{tk:<14}{n:>8}{r:>14.2f}")

In [ ]:
# El español frente al inglés para el MISMO contenido
for tk in TOKENIZADORES:
    es = len(tokens_de(tk, TEXTO_ES))
    en = len(tokens_de(tk, TEXTO_EN))
    print(f"{tk:<12} español={es:>4}  inglés={en:>4}  → el español usa {es / en - 1:+.0%} tokens")

### 🔎 Conclusiones

- Para el mismo contenido, el **español gasta más tokens que el inglés** (la diferencia exacta depende del tokenizador).
- El **código** suele tener pocos caracteres por token: muchos símbolos, indentación y nombres cortados.
- La diferencia entre tokenizadores existe, pero el efecto del **idioma** es consistente.

**Implicación de diseño:** si tu aplicación recibe mucho texto en español, tu factura y tu ventana de contexto se consumen más rápido que con un equivalente en inglés. No es un detalle: se multiplica por miles de llamadas.

## (Opcional) Contar con el servicio de tu proveedor

Los dos proveedores ofrecen un endpoint que cuenta los tokens **exactos** de un texto para un modelo concreto, **sin generar respuesta**:

| Proveedor | Endpoint | Costo |
| --- | --- | --- |
| Anthropic | `count_tokens` | **Gratuito** (con límite de peticiones por minuto) |
| OpenAI | `responses.input_tokens.count` | No genera texto; confirma en su página de precios si se factura |

Necesita tu API key. Compararemos lo que dice `tiktoken` (local) contra lo que dice el proveedor.

In [ ]:
# 💰 FYI: esta celda no genera texto. En Anthropic el conteo es gratuito; en OpenAI confirma en su página de precios si se factura.
if HAY_KEY:
    MODELO_CONTEO = modelo_economico()
    print(f"Conteo del proveedor ({PROVEEDOR}) con {MODELO_CONTEO}\n")
    print(f"{'texto':<10}{'proveedor':>10}{'o200k':>8}{'diferencia':>12}")
    for nombre_texto, texto in TEXTOS.items():
        real = contar_tokens_exacto(texto)
        aprox = len(tokens_de("o200k_base", texto))
        print(f"{nombre_texto:<10}{real:>10}{aprox:>8}{aprox / real - 1:>+12.0%}")
    print("\n(El proveedor cuenta unos pocos tokens extra por la estructura del mensaje.)")
else:
    print("Sin API key: celda omitida. No pasa nada, ya viste el concepto con tiktoken.")

**Lección:** `tiktoken` es una **estimación** útil para el orden de magnitud, pero para presupuestar usa el conteo oficial de tu proveedor o, mejor aún, los tokens que informa cada respuesta.

> Nota: los modelos más recientes pueden usar un tokenizador distinto al de generaciones anteriores y contar **más tokens para el mismo texto**. Es otra razón para medir con el modelo que realmente vas a usar.

---
# Parte 3 · Del token al costo

## La fórmula

$$
\text{costo} = \frac{\text{tokens de entrada} \times P_{\text{entrada}} + \text{tokens de salida} \times P_{\text{salida}}}{1\,000\,000}
$$

donde $P$ es el precio en USD **por millón de tokens**.

Dos cosas que conviene grabarse:

1. **Entrada y salida se cobran distinto.** Los tokens de salida cuestan varias veces más que los de entrada (aquí, 5x).
2. **Precio por token ≠ costo por tarea.** Un modelo caro que resuelve la tarea a la primera puede salir más barato que uno barato que necesita tres intentos. Mide siempre el costo **por tarea completada**.

Ya definimos `costo_usd()` en la Parte 0. Calculemos el costo de **una** llamada que envía nuestro ticket en español y espera ~300 tokens de respuesta, en distintos modelos:

In [ ]:
tokens_in = len(tokens_de("o200k_base", TEXTO_ES))   # aproximación
tokens_out = 300

print(f"Entrada ≈ {tokens_in} tokens · salida estimada = {tokens_out} tokens\n")
print(f"{'modelo':<20}{'USD / llamada':>16}{'USD / 10 000 llamadas':>26}")
for modelo in PRECIOS:
    c = costo_usd(modelo, tokens_in, tokens_out)
    print(f"{modelo:<20}{c:>16.6f}{c * 10_000:>26.2f}")

### 🔎 Observa

Una llamada cuesta fracciones de centavo; **diez mil** llamadas ya son un presupuesto. Y la diferencia entre el modelo más barato y el más caro es de **dos órdenes de magnitud**.

Por eso en el Curso III dedicamos sesiones enteras a *AI FinOps*. Por ahora quédate con el hábito: **antes de llamar, estima; después de llamar, mide**.

### ✍️ Ejercicio rápido

Un equipo de soporte recibe **2 000 tickets al día**. Cada uno envía ~600 tokens al modelo y recibe ~150 de respuesta. ¿Cuánto cuesta **al mes** (30 días) con un modelo económico y con uno caro de la tabla de precios (por ejemplo `claude-haiku-5-5` frente a `claude-opus-5-5`, o `gpt-6-luna` frente a `gpt-6-astra`)? Completa la celda:

In [ ]:
tickets_dia = 2000
tokens_in_ticket = 600
tokens_out_ticket = 150

# TODO: calcula el costo mensual con cada modelo usando costo_usd()
# costo_mensual_haiku = ...
# costo_mensual_opus = ...
# print(...)

---
# Parte 4 · La ventana de contexto

## ¿Qué es?

La **ventana de contexto** es la **memoria de trabajo** del modelo, algo así como su RAM. Ahí caben, **todas juntas**:

- las instrucciones (*system prompt*),
- el historial de la conversación,
- los documentos que le entregas,
- y **la respuesta que genera**.

Todo comparte **el mismo límite de tokens**.

## Tres consecuencias

1. **Todo cuenta**, entrada *y* salida.
2. **Más contexto = más costo y más latencia**, y no siempre mejor calidad (el modelo puede "perderse" en un contexto enorme o lleno de ruido).
3. **No es memoria.** El modelo **no recuerda nada entre llamadas**. Si quieres que sepa algo, se lo vuelves a enviar.

> **Analogía:** es como HTTP, sin estado. Cada petición es independiente; si quieres "sesión", la reenvías tú en cada petición.

Las ventanas de los modelos actuales son de cientos de miles hasta un millón de tokens, pero **esas cifras cambian seguido**: consúltalas en la documentación de tu proveedor y no las memorices.

## La consecuencia que más sorprende: la conversación se encarece sola

Como la API **no tiene estado**, en un chat de varios turnos tienes que **reenviar todo el historial en cada turno**. Entonces:

- turno 1 envía: pregunta 1
- turno 2 envía: pregunta 1 + respuesta 1 + pregunta 2
- turno 3 envía: pregunta 1 + respuesta 1 + pregunta 2 + respuesta 2 + pregunta 3
- …

El tamaño de entrada crece en cada turno y el **costo acumulado crece de forma cuadrática**. Simulémoslo **sin gastar nada**.

In [ ]:
def simular_chat(turnos: int, tokens_pregunta=80, tokens_respuesta=250, tokens_system=400, modelo=None):
    modelo = modelo or modelo_economico()
    historial = tokens_system
    total_entrada = total_salida = 0
    filas = []
    for t in range(1, turnos + 1):
        entrada_este_turno = historial + tokens_pregunta     # lo que se reenvía TODO
        total_entrada += entrada_este_turno
        total_salida += tokens_respuesta
        historial = entrada_este_turno + tokens_respuesta    # crece el historial
        filas.append((t, entrada_este_turno, costo_usd(modelo, total_entrada, total_salida)))
    return filas


print(f"{'turno':>6}{'tokens enviados':>18}{'costo acumulado (USD)':>26}")
for t, ent, acum in simular_chat(20)[::2]:
    print(f"{t:>6}{ent:>18}{acum:>26.6f}   {'█' * (ent // 150)}")

### 🔎 Observa

La barra crece **cada turno** aunque el usuario escriba lo mismo. Un chat de 20 turnos no cuesta "20 veces un turno": cuesta bastante más.

### Cómo se controla esto (spoiler de las próximas sesiones)

- **Comprimir:** resumir el historial viejo, recortar lo irrelevante.
- **Recuperar** solo los fragmentos útiles (esto es **RAG**, Semana 2).
- **Cachear** el inicio del prompt cuando se repite (Sesión 3).
- **Poner un tope** con `max_tokens` y limitar el historial.

> Todo el curso es, en el fondo, una colección de formas de decidir **qué entra a la ventana de contexto**.

---
# Parte 5 · Tu primera llamada a la API

## Anatomía de una llamada

Una llamada a la API tiene tres partes:

| Parte | Contenido |
| --- | --- |
| **Petición** | El modelo, los mensajes con sus roles, y los parámetros (límite de salida, etc.). |
| **Roles** | `system` (o `instructions`) fija el comportamiento general; `user` es la persona; `assistant` es lo que el modelo respondió antes. |
| **Respuesta** | El texto, el uso de tokens (entrada y salida) y el motivo de parada. |

La forma exacta cambia de proveedor a proveedor: **aprende a leer la documentación, no a memorizar campos**. Así se ve la misma llamada "a mano" con cada SDK (nuestro `cliente_llm.py` oculta estas diferencias):

```python
# Anthropic
respuesta = anthropic.Anthropic().messages.create(
    model="claude-haiku-5-5", max_tokens=1024,
    messages=[{"role": "user", "content": "Hola"}],
)
respuesta.content[0].text ; respuesta.usage.input_tokens ; respuesta.stop_reason

# OpenAI (Responses API)
respuesta = openai.OpenAI().responses.create(
    model="gpt-6-luna", max_output_tokens=1024,
    input=[{"role": "user", "content": "Hola"}],
)
respuesta.output_text ; respuesta.usage.input_tokens ; respuesta.status
```

## ¿Cómo obtengo mi API key?

1. Crea una cuenta en la consola de tu proveedor (**Anthropic Platform** u **OpenAI Platform**).
2. Carga un saldo pequeño (con unos pocos dólares alcanza para **todo** el Curso I) o configura facturación.
3. **Configura un límite de gasto mensual** en la consola de tu proveedor. Hazlo ahora.
4. Crea una API key y cópiala en el archivo `.env` de la raíz del repo:
   - Anthropic: `LLM_PROVIDER=anthropic` y `ANTHROPIC_API_KEY=...`
   - OpenAI: `LLM_PROVIDER=openai` y `OPENAI_API_KEY=...`
5. **Nunca** la pegues en un notebook, en un chat o en un commit.

## Elegimos un modelo económico

Para practicar usamos `claude-haiku-5-5` (Anthropic) o `gpt-6-luna` (OpenAI): los dos cuestan US$ 0.10 por millón de tokens de entrada y US$ 0.50 de salida. Por defecto estos modelos "piensan" antes de responder; esos tokens de razonamiento **también se cobran como salida**. Para tareas simples bajamos el esfuerzo con `esfuerzo="minimo"` en nuestro cliente (en Anthropic equivale a `effort: low`; en OpenAI, a razonamiento apagado).

### Estimación previa de esta llamada

Antes de gastar, estimamos: unos 30 tokens de entrada y hasta 1 024 de salida como **tope**. Costo máximo teórico:

In [ ]:
MODELO_BARATO = modelo_economico()
print(f"Modelo: {MODELO_BARATO}")
print(f"Costo MÁXIMO teórico de la llamada siguiente: US$ {costo_usd(MODELO_BARATO, 30, 1024):.6f}")
print("(en la práctica será bastante menos: la respuesta real será de unas decenas de tokens)")

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.0001)
PREGUNTA = "En una sola frase: ¿qué es un token en un modelo de lenguaje?"

if PUEDE_GASTAR:
    respuesta = gasto.registrar(llamar_llm(
        PREGUNTA,
        max_tokens=1024,          # tope de salida: protege tu bolsillo
        esfuerzo="minimo",        # poco "razonamiento": tarea simple
        streaming=False,
    ))
    print("RESPUESTA:\n", respuesta.texto)
    print(f"\nCosto de esta llamada: US$ {respuesta.costo:.6f}")
else:
    print("Celda de pago omitida (sin key o interruptor apagado).")

## Leer la respuesta con calma

La respuesta no es solo texto: es un **objeto** con metadatos. En nuestro cliente (`ResultadoLLM`), igual para los dos proveedores:

- `tokens_entrada` / `tokens_salida` → **lo que se te cobra**. Es tu medidor de costo desde el primer día. Salen del campo de uso (`usage`) que devuelve cada API.
- `stop_reason` → por qué se detuvo: `end_turn` (terminó natural), `max_tokens` (se cortó por tu tope), entre otros.
- `proveedor`, `modelo`, `segundos_total`, `costo`.

Esta celda no hace llamadas: solo inspecciona la respuesta anterior.

In [ ]:
if PUEDE_GASTAR:
    print("proveedor     :", respuesta.proveedor)
    print("modelo        :", respuesta.modelo)
    print("stop_reason   :", respuesta.stop_reason)
    print("tokens entrada:", respuesta.tokens_entrada)
    print("tokens salida :", respuesta.tokens_salida)
    print("\nPara compararlo: tu estimación con tiktoken de la pregunta fue",
          len(tokens_de("o200k_base", PREGUNTA)), "tokens.")
    print("¿Por qué difiere de tokens_entrada? Pista: la estructura del mensaje y el tokenizador del modelo.")

## La API no tiene estado: demostrémoslo

Voy a decirle mi nombre en una llamada y, en **otra llamada independiente**, preguntarle cómo me llamo. Predicción: no lo sabrá, porque la segunda llamada no incluye la primera.

Luego repetimos la segunda llamada **reenviando el historial** y sí lo sabrá. Esa es toda la "memoria" de un chat: **tú reenvías el historial**.

Costo estimado de las dos llamadas: **menos de US$ 0.0005**.

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.0003) · 3 llamadas
def llamar(mensajes):
    r = gasto.registrar(llamar_llm(mensajes, max_tokens=300, esfuerzo="minimo", streaming=False))
    return r.texto


if PUEDE_GASTAR:
    # Llamada A: sin historial
    a = llamar([{"role": "user", "content": "Hola, me llamo Oscar. Responde solo 'Hola'."}])
    # Llamada B: otra llamada independiente, SIN el historial
    b = llamar([{"role": "user", "content": "¿Cómo me llamo?"}])
    print("Sin historial →", b)

    # Llamada C: con el historial reenviado
    c = llamar([
        {"role": "user", "content": "Hola, me llamo Oscar. Responde solo 'Hola'."},
        {"role": "assistant", "content": a},
        {"role": "user", "content": "¿Cómo me llamo?"},
    ])
    print("\nCon historial →", c)
    gasto.resumen()
else:
    print("Celda de pago omitida (sin key o interruptor apagado).")

### 🔎 Qué aprendimos

- Sin historial, el modelo **no sabe** quién eres: la ventana de contexto **no es memoria**.
- Con el historial reenviado sí lo sabe, pero cada mensaje reenviado **vuelve a costar** (lo vimos en la Parte 4).

---
# Parte 6 · Parámetros de generación con un modelo real

## Primer hallazgo: no todos los modelos aceptan `temperature`

Los modelos más recientes **restringen** los parámetros de muestreo (`temperature`, `top_p`, `top_k`), y cada proveedor lo hace a su manera:

| Proveedor | Qué pasa con `temperature` |
| --- | --- |
| **Anthropic** | `claude-haiku-5-5` **rechaza** valores distintos del predeterminado (error 400). El modelo anterior `claude-haiku-4-5` sí los acepta. |
| **OpenAI** | `gpt-6-luna` los acepta **solo con el razonamiento apagado**. Con el razonamiento activo, los rechaza (error 400). OpenAI **no tiene** `top_k`. |

**Un 400 no se cobra**, porque la petición se rechaza antes de generar nada. Es una forma gratuita de ver la lección: *los parámetros de generación son una característica de cada modelo, no un estándar universal*.

> 🧩 **Detalle del SDK:** en la versión 1.x de la librería `anthropic`, `messages.create()` **ya no tiene** los argumentos `temperature`, `top_p` ni `top_k`; si el modelo los acepta, se envían dentro de `extra_body={...}`. Nuestro cliente lo hace por ti. Es un buen ejemplo de por qué **hay que leer la documentación de la versión que instalaste**.

In [ ]:
# 💰 FYI: esta celda llama a la API, pero el error 400 se rechaza antes de generar tokens y no se cobra.
# Esta llamada DEBERÍA fallar con un error 400.
import anthropic, openai

if PUEDE_GASTAR:
    try:
        if PROVEEDOR == "anthropic":
            # claude-haiku-5-5 rechaza temperature
            llamar_llm("Hola", modelo="claude-haiku-5-5", max_tokens=50, streaming=False, temperature=0.2, max_reintentos=0)
        else:
            # con razonamiento activo ("bajo"), OpenAI rechaza temperature
            llamar_llm("Hola", max_tokens=50, streaming=False, esfuerzo="bajo", temperature=0.2, max_reintentos=0)
        print("Sorpresa: el modelo aceptó temperature (la documentación puede haber cambiado).")
    except (anthropic.BadRequestError, openai.BadRequestError) as e:
        print("Como esperábamos, error 400:")
        print("  ", getattr(e, "message", e))
else:
    print("Celda omitida (sin key o interruptor apagado).")

## Segundo hallazgo: usar un modelo (o una configuración) que sí acepta los parámetros

Para experimentar con la aleatoriedad necesitamos que el modelo acepte `temperature`:

- **Anthropic:** usamos **`claude-haiku-4-5`** (generación anterior, US$ 1 / US$ 5 por millón de tokens).
- **OpenAI:** usamos el **mismo modelo** con el razonamiento apagado (`esfuerzo="minimo"`).

> En la práctica profesional, esto es un criterio de selección de modelo: si tu caso necesita controlar la aleatoriedad, verifica que el modelo exponga esos parámetros. Si no los expone, controlas el comportamiento con el prompt (Sesión 3) y con validación (Sesión 2).

### Experimento A: ¿cuánto varía la respuesta con temperature baja vs alta?

Enviaremos el **mismo prompt 5 veces** con `temperature=0` y 5 veces con `temperature=1`. Es una tarea creativa corta (un nombre), con `max_tokens=30`.

Costo máximo: 10 llamadas × (≈30 tokens de entrada, ≤30 de salida) ≈ **US$ 0.002**.

In [ ]:
MODELO_PARAMS = modelo_muestreo()
PROMPT_NOMBRE = "Dame UN nombre creativo para un servidor de bases de datos. Responde solo con el nombre."

print(f"Modelo para los experimentos: {MODELO_PARAMS}")
print(f"Costo máximo estimado del experimento A: US$ {10 * costo_usd(MODELO_PARAMS, 40, 30):.5f}")

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.001) · 10 llamadas
def preguntar_param(prompt, n=5, **parametros):
    """Mismo prompt `n` veces con los parámetros de muestreo indicados (temperature, top_p, top_k)."""
    salidas = []
    for _ in range(n):
        r = gasto.registrar(llamar_llm(
            prompt, modelo=MODELO_PARAMS, max_tokens=30, streaming=False,
            esfuerzo="minimo",       # en OpenAI, razonamiento apagado: obligatorio para usar temperature/top_p
            **parametros,
        ))
        salidas.append(r.texto.strip())
    return salidas


resultados = {}
if PUEDE_GASTAR:
    for t in (0.0, 1.0):
        resultados[t] = preguntar_param(PROMPT_NOMBRE, temperature=t)
        print(f"temperature={t}:")
        for s in resultados[t]:
            print("   ", s)
    gasto.resumen()
else:
    print("Celda de pago omitida (sin key o interruptor apagado).")

### 🔎 Observa

- Con `temperature=1.0` esperamos nombres **distintos** en cada intento.
- Con `temperature=0` esperamos **muy poca variación**… pero fíjate: ¿salieron **idénticos**? A veces sí, a veces no. `temperature=0` reduce la variación, **no la elimina**.

Contemos cuántas respuestas distintas hubo en cada caso:

In [ ]:
if resultados:
    for t, salidas in resultados.items():
        print(f"temperature={t}: {len(set(salidas))} respuestas distintas de {len(salidas)}")

### Experimento B: `max_tokens` corto → respuesta cortada

`max_tokens` es un **tope duro**. Si el modelo aún no terminó cuando llega al tope, la respuesta **se corta a media frase** y `stop_reason` vale `"max_tokens"`. No es un error: es el comportamiento esperado, y es la forma más directa de **limitar el costo**.

Una sola llamada, con tope de 20 tokens. Costo: **menos de US$ 0.0002**.

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.0001) · 1 llamada
if PUEDE_GASTAR:
    r = gasto.registrar(llamar_llm(
        "Explica en un párrafo qué es una ventana de contexto.",
        modelo=MODELO_PARAMS, max_tokens=20, streaming=False, esfuerzo="minimo",
    ))
    print(r.texto)
    print("\nstop_reason:", r.stop_reason, "| tokens de salida:", r.tokens_salida)
else:
    print("Celda de pago omitida (sin key o interruptor apagado).")

### Experimento C: `top_p` y `top_k`

`temperature` y `top_p` **no se envían juntos** (con esta generación de modelos conviene ajustar uno u otro, no ambos). Probamos `top_p` bajo por un lado y `top_k` por otro. **OpenAI no tiene `top_k`**: nuestro cliente lo ignora, así que con ese proveedor la segunda prueba solo repite una llamada normal. Cada uno 3 llamadas. Costo: **menos de US$ 0.001**.

In [ ]:
# 💰 FYI: esta celda consume dinero (≈ US$ 0.0007) · 6 llamadas
if PUEDE_GASTAR:
    print("top_p=0.1:", preguntar_param(PROMPT_NOMBRE, n=3, top_p=0.1))
    print("top_k=1  :", preguntar_param(PROMPT_NOMBRE, n=3, top_k=1), "(OpenAI no tiene top_k: se ignora)" if PROVEEDOR == "openai" else "")
    gasto.resumen()
else:
    print("Celda de pago omitida (sin key o interruptor apagado).")

### ✅ Respuesta a la pregunta de la Parte 1

- **Clasificar tickets / extraer datos:** temperatura **baja**. Quieres consistencia.
- **Proponer nombres / lluvia de ideas:** temperatura **más alta**. Quieres variedad.
- Y si el modelo **no** deja tocar la temperatura, controlas la consistencia con prompts claros, ejemplos y validación de la salida.

---
# Parte 7 · Determinista vs probabilístico, y cuándo NO usar un LLM

## El cambio mental más importante del día

| | Software tradicional | Sistema con LLM |
| --- | --- | --- |
| Misma entrada | Misma salida | La salida **puede variar** |
| Un error es… | Una excepción, un código de error | Una respuesta que **suena bien pero está mal** |
| Se prueba con… | `assert resultado == esperado` | Casos de **evaluación** y métricas |
| "Funcionó una vez" | Suele significar que funciona | **No significa nada**; puede fallar la próxima |

Por eso este curso insiste en **evaluar** (Semana 3). Que funcione una vez en la demo no significa que funcione.

### Demostración con tus propios datos de la Parte 6

Lo que acabas de medir con `temperature=0` ya es evidencia: ¿puedes escribir un `assert` que pase *siempre*?

In [ ]:
if resultados:
    salidas_t0 = resultados[0.0]
    try:
        assert len(set(salidas_t0)) == 1, "Las 5 respuestas deberían ser idénticas con temperature=0"
        print("Esta vez las 5 respuestas fueron idénticas. Ejecuta el experimento A otra vez: ¿se mantiene?")
    except AssertionError as e:
        print("El assert falló:", e)
        print(salidas_t0)
else:
    print("Sin datos de la Parte 6 (celdas de pago omitidas). Idea: un assert exacto sobre un LLM es frágil.")

## Cuándo NO usar un LLM

Un buen AI Engineer también sabe decir que **no**. Evita el LLM cuando:

- Hay **reglas exactas** (validar un formato, calcular un monto): eso es código.
- La decisión debe ser **explicable y reproducible** (auditoría, cumplimiento).
- Necesitas **millones de llamadas baratas y rápidas**.
- Hay **datos sensibles** y no sabes dónde terminan.

> **Regla práctica:** si un `if` o una expresión regular lo resuelve, empieza por ahí. No usarías un LLM para decidir si un certificado expiró; **sí** para redactar el resumen del incidente.

### Ejemplo: el certificado

Decidir si un certificado expiró es una resta de fechas. Es **gratis, instantáneo, exacto y reproducible**:

In [ ]:
from datetime import date

def certificado_expirado(fecha_vencimiento: date, hoy: date | None = None) -> bool:
    hoy = hoy or date.today()
    return fecha_vencimiento < hoy


print(certificado_expirado(date(2026, 10, 12), hoy=date(2026, 10, 13)))   # True, siempre
print(certificado_expirado(date(2026, 12, 31), hoy=date(2026, 10, 13)))   # False, siempre

Un LLM aquí sería más lento, más caro y **a veces se equivocaría**. En cambio, para **redactar** el resumen del incidente para la gerencia, el LLM sí aporta:

```
Entrada (datos estructurados, calculados por código):
  servidor=app-prod-03, certificado_expirado=True, impacto=facturación, hora=02:15
Salida (redactada por el LLM):
  "A las 02:15 el servicio de facturación dejó de responder por un certificado vencido..."
```

**Patrón de diseño que verás todo el curso:** el código decide lo exacto; el LLM maneja lo ambiguo y el lenguaje.

### 🗣️ Para discutir en grupo

¿Cómo probarían hoy un chatbot que responde las políticas de la empresa? Anota 2 o 3 ideas. Las retomamos en la Semana 3 (*evaluación*).

---
# Parte 8 · Cierre

## Cuánto gastaste hoy

Ejecuta esta celda para ver el resumen real de tu gasto en la sesión:

In [ ]:
gasto.resumen()
for r in gasto.resultados:
    print(f"  {r.modelo:<18} in={r.tokens_entrada:>4} out={r.tokens_salida:>4}  US$ {r.costo:.6f}")
print("\nContrasta este número con la estimación del inicio. Revisa también el consumo en la consola de tu proveedor.")

## 📌 Lo que te llevas

1. Un LLM **predice el siguiente token**; no es una base de datos ni un buscador.
2. El **token es la unidad de costo y de límite**. El idioma y el tokenizador cambian cuántos usas.
3. La **ventana de contexto se comparte y se paga**, y **no es memoria**: tú reenvías el historial.
4. Es un sistema **probabilístico**: se **evalúa**, no se prueba con `assertEqual`.
5. Los **parámetros de generación** (temperature, top_p, top_k, max_tokens) existen, pero **cada modelo decide** cuáles acepta.
6. Si un `if` lo resuelve, **no uses un LLM**.

## ✅ Autoevaluación

Responde sin mirar arriba (no hay calificación, es para ti):

1. ¿Por qué el español gasta más tokens que el inglés?
2. ¿Qué dos cosas se cobran distinto en una llamada?
3. ¿Por qué un chat de 20 turnos cuesta más que 20 veces un turno?
4. ¿Qué garantiza `temperature=0`? ¿Qué **no** garantiza?
5. Da un ejemplo de tarea donde NO usarías un LLM.

## 🚀 Retos opcionales

- **Reto 1:** Usa la calculadora de la práctica (`practica/calculadora_tokens.py`) con un texto tuyo (un log, un correo). ¿Cuál es tu ratio de caracteres por token?
- **Reto 2:** Modifica `simular_chat()` para que el historial solo conserve los últimos 6 turnos. ¿Cuánto ahorras en 20 turnos?
- **Reto 3:** Cambia el idioma de un prompt y mide la diferencia de costo para 10 000 llamadas.

## 📖 Glosario

| Término | Significado |
| --- | --- |
| **LLM** | Modelo de lenguaje grande: predice el siguiente token. |
| **Token** | Fragmento de texto; unidad de costo y de límite. |
| **Tokenizador** | La tabla que parte el texto en tokens; propia de cada modelo. |
| **Ventana de contexto** | Máximo de tokens (entrada + salida) que el modelo maneja en una llamada. |
| **Transformer** | Arquitectura de los LLM actuales, basada en capas de atención. |
| **temperature / top_p / top_k** | Parámetros que controlan qué tan aleatoria es la elección del siguiente token. |
| **max_tokens** | Tope de longitud de la respuesta. |
| **Stateless (sin estado)** | La API no recuerda llamadas previas; tú reenvías el historial. |

## ➡️ Siguiente sesión

**Sesión 2 · Ecosistema de modelos y consumo vía API.** Elegiremos modelos con criterio, haremos *streaming*, manejaremos errores con reintentos y pediremos salida estructurada (JSON validado). Trae tu API key probada.

**Práctica de esta sesión:** la *Calculadora de tokens y costo* está en `practica/`.